In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CSV FILE UPLOAD FUNCTION
# ============================================

def upload_csv_file():
    """
    Upload CSV file in Google Colab or local environment
    """
    try:
        # For Google Colab
        from google.colab import files
        import io

        print("=" * 70)
        print("CSV FILE UPLOAD")
        print("=" * 70)
        print("\nPlease upload your heart disease dataset (CSV format)")
        print("Expected columns should include: Age, RestingBP, Cholesterol,")
        print("FastingBS, MaxHR, Oldpeak, HeartDisease, and other features")
        print("\nClick 'Choose Files' button below to upload...")

        uploaded = files.upload()

        # Get the uploaded file name
        file_name = list(uploaded.keys())[0]

        # Load the dataset
        df = pd.read_csv(io.BytesIO(uploaded[file_name]))

        print(f"\nFile '{file_name}' uploaded successfully!")
        return df

    except ImportError:
        # For local environment
        import os

        print("=" * 70)
        print("LOCAL FILE SELECTION")
        print("=" * 70)

        print("\nCurrent directory:", os.getcwd())
        print("\nAvailable CSV files:")

        # List CSV files
        csv_files = [f for f in os.listdir('.') if f.lower().endswith('.csv')]

        if csv_files:
            for i, file in enumerate(csv_files, 1):
                print(f"  {i}. {file}")

            choice = input("\nEnter file number or path: ").strip()

            if choice.isdigit() and 1 <= int(choice) <= len(csv_files):
                file_name = csv_files[int(choice) - 1]
                df = pd.read_csv(file_name)
            else:
                # Assume it's a file path
                df = pd.read_csv(choice)
        else:
            print("No CSV files found. Please place your CSV file in the current directory.")
            print("Current directory:", os.getcwd())
            return None

        return df

# ============================================
# MAIN PROGRAM STARTS HERE
# ============================================

print("\n" + "=" * 70)
print("HEART DISEASE RISK PREDICTION SYSTEM")
print("=" * 70)

# Step 1: Upload CSV file
df = upload_csv_file()

if df is None:
    print("No dataset loaded. Exiting program.")
    exit()

print(f"\nDataset loaded successfully!")
print(f"Number of records: {len(df)}")
print(f"Number of features: {len(df.columns)}")

# Check if HeartDisease column exists
if 'HeartDisease' not in df.columns:
    print("\nERROR: 'HeartDisease' column not found in dataset.")
    print("Please ensure your CSV file contains a 'HeartDisease' column (0/1 values).")
    exit()

print(f"\nDataset Preview:")
print(df.head())

# ============================================
# DATA PREPARATION
# ============================================

print("\n" + "=" * 70)
print("DATA PREPARATION")
print("=" * 70)

# Prepare features and target
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Convert boolean columns to integer if they exist
bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)
    print(f"Converted {len(bool_cols)} boolean columns to integer")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nData split:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

# Identify numerical features for scaling
numerical_features = []
for col in ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']:
    if col in X.columns:
        numerical_features.append(col)

print(f"\nNumerical features to scale: {numerical_features}")

# Scale numerical features
if numerical_features:
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
    X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

    print("Numerical features scaled successfully")
else:
    X_train_scaled = X_train
    X_test_scaled = X_test
    scaler = None
    print("No numerical features found for scaling")

# ============================================
# MODEL TRAINING
# ============================================

print("\n" + "=" * 70)
print("MODEL TRAINING")
print("=" * 70)

# Use Gradient Boosting for probability predictions
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print(f"Training Accuracy: {model.score(X_train_scaled, y_train):.3f}")
print(f"Testing Accuracy: {model.score(X_test_scaled, y_test):.3f}")

# Make predictions on test set
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"\nModel Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")

# ============================================
# PREDICTION FUNCTION
# ============================================

def predict_heart_disease_risk(patient_data):
    """
    Predict heart disease risk for a new patient
    Returns: (prediction, risk_percentage)
    """
    # Create DataFrame from patient data
    patient_df = pd.DataFrame([patient_data])

    # Ensure all columns are present
    for col in X.columns:
        if col not in patient_df.columns:
            patient_df[col] = 0

    # Reorder columns to match training data
    patient_df = patient_df[X.columns]

    # Convert boolean values if present
    for col in bool_cols:
        if col in patient_df.columns:
            patient_df[col] = int(patient_df[col])

    # Scale numerical features if scaler exists
    if scaler is not None and numerical_features:
        patient_df[numerical_features] = scaler.transform(patient_df[numerical_features])

    # Get probability and prediction
    probability = model.predict_proba(patient_df)[0][1]
    risk_percentage = probability * 100
    prediction = model.predict(patient_df)[0]

    return prediction, risk_percentage

# ============================================
# EXAMPLE PATIENTS DEMONSTRATION
# ============================================

print("\n" + "=" * 70)
print("EXAMPLE PATIENT ASSESSMENTS")
print("=" * 70)

# Example 1: High Risk Patient
print("\nEXAMPLE 1: HIGH RISK PATIENT")
print("-" * 50)

high_risk_patient = {
    'Age': 65,
    'RestingBP': 180,
    'Cholesterol': 350,
    'FastingBS': 1,
    'MaxHR': 100,
    'Oldpeak': 3.5,
    'Sex_M': 1,
    'ChestPainType_ATA': 0,
    'ChestPainType_NAP': 1,
    'ChestPainType_TA': 0,
    'RestingECG_Normal': 0,
    'RestingECG_ST': 1,
    'ExerciseAngina_Y': 1,
    'ST_Slope_Flat': 1,
    'ST_Slope_Up': 0
}

print("Patient Profile:")
print(f"  Age: {high_risk_patient['Age']} years")
print(f"  Blood Pressure: {high_risk_patient['RestingBP']} mmHg")
print(f"  Cholesterol: {high_risk_patient['Cholesterol']} mg/dL")
print(f"  Fasting Blood Sugar: {'High (>120 mg/dL)' if high_risk_patient['FastingBS'] == 1 else 'Normal'}")
print(f"  Max Heart Rate: {high_risk_patient['MaxHR']} bpm")
print(f"  ST Depression: {high_risk_patient['Oldpeak']}")

prediction, risk = predict_heart_disease_risk(high_risk_patient)
print(f"\nRISK ASSESSMENT:")
print(f"  Heart Disease Prediction: {'YES' if prediction == 1 else 'NO'}")
print(f"  Risk Probability: {risk:.1f}%")

if risk > 70:
    print(f"  RISK LEVEL: VERY HIGH - Immediate medical attention needed")
elif risk > 40:
    print(f"  RISK LEVEL: HIGH - Consult cardiologist")
elif risk > 20:
    print(f"  RISK LEVEL: MODERATE - Lifestyle changes recommended")
else:
    print(f"  RISK LEVEL: LOW - Maintain healthy lifestyle")

# Example 2: Low Risk Patient
print("\n\nEXAMPLE 2: LOW RISK PATIENT")
print("-" * 50)

low_risk_patient = {
    'Age': 35,
    'RestingBP': 120,
    'Cholesterol': 180,
    'FastingBS': 0,
    'MaxHR': 170,
    'Oldpeak': 0.0,
    'Sex_M': 0,
    'ChestPainType_ATA': 1,
    'ChestPainType_NAP': 0,
    'ChestPainType_TA': 0,
    'RestingECG_Normal': 1,
    'RestingECG_ST': 0,
    'ExerciseAngina_Y': 0,
    'ST_Slope_Flat': 0,
    'ST_Slope_Up': 1
}

print("Patient Profile:")
print(f"  Age: {low_risk_patient['Age']} years")
print(f"  Blood Pressure: {low_risk_patient['RestingBP']} mmHg")
print(f"  Cholesterol: {low_risk_patient['Cholesterol']} mg/dL")
print(f"  Fasting Blood Sugar: {'High (>120 mg/dL)' if low_risk_patient['FastingBS'] == 1 else 'Normal'}")
print(f"  Max Heart Rate: {low_risk_patient['MaxHR']} bpm")
print(f"  ST Depression: {low_risk_patient['Oldpeak']}")

prediction, risk = predict_heart_disease_risk(low_risk_patient)
print(f"\nRISK ASSESSMENT:")
print(f"  Heart Disease Prediction: {'YES' if prediction == 1 else 'NO'}")
print(f"  Risk Probability: {risk:.1f}%")

if risk > 70:
    print(f"  RISK LEVEL: VERY HIGH - Immediate medical attention needed")
elif risk > 40:
    print(f"  RISK LEVEL: HIGH - Consult cardiologist")
elif risk > 20:
    print(f"  RISK LEVEL: MODERATE - Lifestyle changes recommended")
else:
    print(f"  RISK LEVEL: LOW - Maintain healthy lifestyle")

# ============================================
# INTERACTIVE PREDICTION
# ============================================

print("\n" + "=" * 70)
print("INTERACTIVE HEART DISEASE RISK ASSESSMENT")
print("=" * 70)

print("\nEnter patient details for risk assessment")
print("(Press Enter to use default values or type 'exit' to quit)")

try:
    print("\nPatient Information:")
    age = int(input("Age (years) [55]: ") or "55")
    resting_bp = int(input("Resting Blood Pressure (mmHg) [140]: ") or "140")
    cholesterol = int(input("Cholesterol (mg/dL) [240]: ") or "240")
    fasting_bs = int(input("Fasting Blood Sugar (0=Normal, 1=High>120) [0]: ") or "0")
    max_hr = int(input("Maximum Heart Rate (bpm) [150]: ") or "150")
    oldpeak = float(input("ST Depression (Oldpeak) [1.0]: ") or "1.0")

    sex = input("Sex (M/F) [M]: ").upper() or "M"
    sex_m = 1 if sex == 'M' else 0

    print("\nChest Pain Type:")
    print("  1. Atypical Angina (ATA)")
    print("  2. Non-Anginal Pain (NAP)")
    print("  3. Typical Angina (TA)")
    chest_pain = int(input("Enter choice (1/2/3) [1]: ") or "1")

    chest_pain_ata = 1 if chest_pain == 1 else 0
    chest_pain_nap = 1 if chest_pain == 2 else 0
    chest_pain_ta = 1 if chest_pain == 3 else 0

    print("\nResting ECG:")
    print("  1. Normal")
    print("  2. ST-T wave abnormality")
    ecg = int(input("Enter choice (1/2) [1]: ") or "1")

    ecg_normal = 1 if ecg == 1 else 0
    ecg_st = 1 if ecg == 2 else 0

    exercise_input = input("Exercise Induced Angina? (Y/N) [N]: ").upper() or "N"
    exercise_angina = 1 if exercise_input == 'Y' else 0

    print("\nST Slope:")
    print("  1. Flat")
    print("  2. Up")
    st_slope = int(input("Enter choice (1/2) [2]: ") or "2")

    st_slope_flat = 1 if st_slope == 1 else 0
    st_slope_up = 1 if st_slope == 2 else 0

    # Create patient dictionary
    user_patient = {
        'Age': age,
        'RestingBP': resting_bp,
        'Cholesterol': cholesterol,
        'FastingBS': fasting_bs,
        'MaxHR': max_hr,
        'Oldpeak': oldpeak,
        'Sex_M': sex_m,
        'ChestPainType_ATA': chest_pain_ata,
        'ChestPainType_NAP': chest_pain_nap,
        'ChestPainType_TA': chest_pain_ta,
        'RestingECG_Normal': ecg_normal,
        'RestingECG_ST': ecg_st,
        'ExerciseAngina_Y': exercise_angina,
        'ST_Slope_Flat': st_slope_flat,
        'ST_Slope_Up': st_slope_up
    }

    # Add any missing columns that might be in the dataset
    for col in X.columns:
        if col not in user_patient:
            user_patient[col] = 0

    print("\n" + "=" * 50)
    print("PATIENT PROFILE SUMMARY")
    print("=" * 50)

    print(f"\nBasic Information:")
    print(f"  Age: {age} years")
    print(f"  Sex: {'Male' if sex_m == 1 else 'Female'}")

    print(f"\nVital Signs:")
    print(f"  Blood Pressure: {resting_bp} mmHg")
    print(f"  Cholesterol: {cholesterol} mg/dL")
    print(f"  Fasting Blood Sugar: {'High (>120 mg/dL)' if fasting_bs == 1 else 'Normal'}")
    print(f"  Max Heart Rate: {max_hr} bpm")
    print(f"  ST Depression: {oldpeak}")

    print(f"\nMedical History:")
    chest_pain_types = []
    if chest_pain_ata: chest_pain_types.append("Atypical Angina")
    if chest_pain_nap: chest_pain_types.append("Non-Anginal Pain")
    if chest_pain_ta: chest_pain_types.append("Typical Angina")
    print(f"  Chest Pain: {', '.join(chest_pain_types) if chest_pain_types else 'None reported'}")

    ecg_type = "Normal" if ecg_normal else "ST-T wave abnormality"
    print(f"  Resting ECG: {ecg_type}")
    print(f"  Exercise Angina: {'Yes' if exercise_angina else 'No'}")

    st_slope_type = "Flat" if st_slope_flat else "Up"
    print(f"  ST Slope: {st_slope_type}")

    # Make prediction
    print("\n" + "=" * 50)
    print("HEART DISEASE RISK PREDICTION")
    print("=" * 50)

    prediction, risk = predict_heart_disease_risk(user_patient)

    print(f"\nPrediction Results:")
    print(f"  Heart Disease: {'YES' if prediction == 1 else 'NO'}")
    print(f"  Risk Probability: {risk:.1f}%")

    # Risk interpretation
    print(f"\nRisk Interpretation:")
    if risk > 80:
        print(f"  CRITICAL RISK ({risk:.1f}%)")
        print("  - Immediate medical attention required")
        print("  - High probability of heart disease")
        print("  - Consult cardiologist urgently")
    elif risk > 60:
        print(f"  HIGH RISK ({risk:.1f}%)")
        print("  - High probability of heart disease")
        print("  - Schedule cardiologist appointment")
        print("  - Consider lifestyle changes")
    elif risk > 40:
        print(f"  MODERATE RISK ({risk:.1f}%)")
        print("  - Moderate probability of heart disease")
        print("  - Regular checkups recommended")
        print("  - Improve diet and exercise")
    elif risk > 20:
        print(f"  LOW-MODERATE RISK ({risk:.1f}%)")
        print("  - Low to moderate probability")
        print("  - Maintain healthy lifestyle")
        print("  - Regular monitoring advised")
    else:
        print(f"  LOW RISK ({risk:.1f}%)")
        print("  - Very low probability of heart disease")
        print("  - Continue healthy lifestyle")
        print("  - Regular checkups still recommended")

    # Show risk factors
    print(f"\nIdentified Risk Factors:")

    risk_factors = []

    if age > 50: risk_factors.append(f"Age ({age} years - over 50)")
    if resting_bp > 140: risk_factors.append(f"High BP ({resting_bp} mmHg)")
    if cholesterol > 240: risk_factors.append(f"High Cholesterol ({cholesterol} mg/dL)")
    if fasting_bs == 1: risk_factors.append("High Blood Sugar")
    if max_hr < 120: risk_factors.append(f"Low Max Heart Rate ({max_hr} bpm)")
    if oldpeak > 2.0: risk_factors.append(f"High ST Depression ({oldpeak})")
    if exercise_angina: risk_factors.append("Exercise Induced Angina")

    if risk_factors:
        for i, factor in enumerate(risk_factors, 1):
            print(f"  {i}. {factor}")
    else:
        print("  No major risk factors identified")

    print("\n" + "=" * 50)

except ValueError as e:
    print(f"\nInput Error: Please enter valid numerical values")
    print(f"Error details: {e}")
except Exception as e:
    print(f"\nUnexpected Error: {e}")
    print("Please try again with valid inputs")

# ============================================
# MODEL PERFORMANCE VISUALIZATION
# ============================================

print("\n" + "=" * 70)
print("MODEL PERFORMANCE VISUALIZATION")
print("=" * 70)

# Create performance plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='darkblue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nModel Performance Summary:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"AUC-ROC: {roc_auc:.3f}")
print(f"Confusion Matrix:")
print(cm)

print("\n" + "=" * 70)
print("HEART DISEASE RISK ASSESSMENT COMPLETED")
print("=" * 70)


HEART DISEASE RISK PREDICTION SYSTEM
CSV FILE UPLOAD

Please upload your heart disease dataset (CSV format)
Expected columns should include: Age, RestingBP, Cholesterol,
FastingBS, MaxHR, Oldpeak, HeartDisease, and other features

Click 'Choose Files' button below to upload...


Saving heart_processed.csv to heart_processed (2).csv

File 'heart_processed (2).csv' uploaded successfully!

Dataset loaded successfully!
Number of records: 918
Number of features: 16

Dataset Preview:
   Age  RestingBP  Cholesterol  FastingBS  MaxHR  Oldpeak  HeartDisease  \
0   40        140          289          0    172      0.0             0   
1   49        160          180          0    156      1.0             1   
2   37        130          283          0     98      0.0             0   
3   48        138          214          0    108      1.5             1   
4   54        150          195          0    122      0.0             0   

   Sex_M  ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  \
0   True               True              False             False   
1  False              False               True             False   
2   True               True              False             False   
3  False              False              False             False   
4   Tr